# XQuality Prefix Stop-After-Layer Evaluation — Exact CLI-Compatible Evaluation

This notebook evaluates the generated prefix stop-after-layer outputs using the **same NeoOLAF evaluation path** as the gold-evaluation notebook:

```bash
python -m neoolaf.evaluation evaluate \
  --dataset xquality \
  --method neoolaf \
  --profile xquality_relaxed_recall \
  --input <export_or_prefix_folder> \
  --gold data/XQuality/Examples/XQuality_all_triplets_flat_en.json
```

Important: this notebook evaluates **export folders** (`kg.json`, `kg_local.json`, or `kg_inferred.json`) with `evaluate_run(...)`, not direct `evaluate_extraction(...)`. This matters because `evaluate_run(...)` applies XQuality-specific canonicalization for the `xquality_relaxed_recall` profile.


In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
import subprocess
from pathlib import Path
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)


## 1. Locate the NeoOLAF repository

In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for c in candidates:
        if (c / "src" / "neoolaf").exists() and (c / "examples" / "XQualityMachine32").exists():
            return c
    # Common fallback if launched from examples/XQualityMachine32
    for c in candidates:
        if c.name.lower() == "neoolaf":
            return c
    raise RuntimeError(f"Could not find NeoOLAF repo root from {start}")

ROOT = find_repo_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)
print("Python:", sys.executable)


## 2. Configuration

The profile is intentionally set to `xquality_relaxed_recall`, because this is the profile used in the gold-evaluation notebook that produced the higher final score (`relation_f1 ≈ 0.6108`).

In [ ]:
# Same gold-evaluation configuration as the no-gold / gold notebook.
DOCUMENT_PROFILE = "xquality_relaxed_recall"

GOLD_JSON = ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"
SEED_ONTOLOGY = ROOT / "data" / "ontology" / "ContextOntology-COInd4.owl"

RUNS_ROOT = ROOT / "examples" / "XQualityMachine32" / "runs" / "xquality_machine32"
LAYER12_RUN_DIR = RUNS_ROOT / "layer12_from_l11"
NATIVE_EXPORT_DIR = LAYER12_RUN_DIR / "exports"

# Prefix generated outputs from the stop-after-layer notebook.
PREFIX_OUTPUT_ROOT = (
    ROOT / "examples" / "XQualityMachine32" / "runs" /
    "prefix_stop_after_layer_llm_finalization_exact_eval"
)

# Output of this exact evaluation notebook.
EVAL_OUTPUT_ROOT = (
    ROOT / "examples" / "XQualityMachine32" / "runs" /
    "prefix_stop_after_layer_exact_cli_profile_eval"
)

EVAL_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DOCUMENT_PROFILE:", DOCUMENT_PROFILE)
print("GOLD_JSON:", GOLD_JSON, "| exists:", GOLD_JSON.exists())
print("SEED_ONTOLOGY:", SEED_ONTOLOGY, "| exists:", SEED_ONTOLOGY.exists())
print("PREFIX_OUTPUT_ROOT:", PREFIX_OUTPUT_ROOT, "| exists:", PREFIX_OUTPUT_ROOT.exists())
print("NATIVE_EXPORT_DIR:", NATIVE_EXPORT_DIR, "| exists:", NATIVE_EXPORT_DIR.exists())

assert GOLD_JSON.exists(), f"Gold JSON not found: {GOLD_JSON}"
assert PREFIX_OUTPUT_ROOT.exists(), f"Prefix output root not found: {PREFIX_OUTPUT_ROOT}"
assert NATIVE_EXPORT_DIR.exists(), f"Native export dir not found: {NATIVE_EXPORT_DIR}"


## 3. Import the exact NeoOLAF evaluation runner

This uses `evaluate_run(...)`, the same runner used by `python -m neoolaf.evaluation evaluate`.

Do **not** replace this with a local P/R/F1 implementation if you want scores comparable to the original gold-evaluation notebook.


In [ ]:
from neoolaf.evaluation.runners.evaluate_run import evaluate_run
from neoolaf.evaluation.profiles.registry import get_profile

profile = get_profile(DOCUMENT_PROFILE)
print("Loaded profile:", profile.name)
print("Profile gt_guided_canonicalization:", getattr(profile, "gt_guided_canonicalization", None))
print("Profile use_alias_maps:", getattr(profile, "use_alias_maps", None))
print("Profile use_alarm_number_anchoring:", getattr(profile, "use_alarm_number_anchoring", None))


## 4. Discover completed prefix output folders

A valid folder must contain one of:

- `kg.json`
- `kg_local.json`
- `kg_inferred.json`

The adapter used by NeoOLAF supports these names.


In [ ]:
def has_kg_file(folder: Path) -> bool:
    return any((folder / name).exists() for name in ["kg_inferred.json", "kg_local.json", "kg.json"])

def find_kg_file(folder: Path) -> Path | None:
    for name in ["kg_inferred.json", "kg_local.json", "kg.json"]:
        p = folder / name
        if p.exists():
            return p
    return None

def parse_stop_index(folder_name: str) -> int | None:
    m = re.search(r"prefix_stop_after_(\d+)_", folder_name)
    if m:
        return int(m.group(1))
    return None

prefix_folders = []
for folder in sorted(PREFIX_OUTPUT_ROOT.iterdir()):
    if not folder.is_dir():
        continue
    if not folder.name.startswith("prefix_stop_after_"):
        continue
    kg = find_kg_file(folder)
    if kg is None:
        continue
    # If a layer was interrupted before writing a usable kg.json, skip it.
    try:
        data = json.loads(kg.read_text(encoding="utf-8"))
        triple_count = len(data.get("triples", [])) if isinstance(data, dict) else 0
    except Exception as e:
        print("Skipping unreadable kg:", kg, repr(e))
        continue
    prefix_folders.append({
        "series": "prefix_stop_after_layer_generated_output",
        "layer_name": folder.name,
        "stop_index": parse_stop_index(folder.name),
        "input_folder": folder,
        "kg_file": kg,
        "triple_count": triple_count,
    })

prefix_df = pd.DataFrame(prefix_folders).sort_values(["stop_index", "layer_name"])
display(prefix_df)

print("Found prefix folders:", len(prefix_df))


## 5. Evaluation helpers

This calls `evaluate_run(...)` for every prefix output folder and for the native final NeoOLAF export.

The native final export is evaluated with the exact same code path and should reproduce the known final score around:

```text
relation_precision = 1.0
relation_recall ≈ 0.4396
relation_f1 ≈ 0.6108
```

If it does not, the paths/profile are not the same as the original gold-evaluation notebook.


In [ ]:
def safe_name(name: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(name)).strip("_")

def run_exact_eval_for_folder(
    *,
    input_folder: Path,
    out_name: str,
    run_id: str,
    series: str,
    layer_name: str,
    stop_index: int | None,
    triple_count: int | None,
) -> dict:
    out_dir = EVAL_OUTPUT_ROOT / safe_name(out_name)
    out_dir.mkdir(parents=True, exist_ok=True)

    summary = evaluate_run(
        dataset="xquality",
        method="neoolaf",
        profile_name=DOCUMENT_PROFILE,
        gold_path=GOLD_JSON,
        output_dir=out_dir,
        input_path=input_folder,
        ontology_path=SEED_ONTOLOGY if SEED_ONTOLOGY.exists() else None,
        run_id=run_id,
    )

    extraction = summary.get("extraction", {})
    ent = extraction.get("entity", {})
    rel = extraction.get("relation", {})
    counts = extraction.get("counts", {})

    row = {
        "series": series,
        "layer_name": layer_name,
        "stop_index": stop_index,
        "evaluation_mode": "neoolaf_evaluate_run_cli_path",
        "profile": summary.get("profile", DOCUMENT_PROFILE),
        "triple_count": triple_count,
        "input_folder": str(input_folder),
        "eval_output_dir": str(out_dir),
        "entity_tp": ent.get("tp"),
        "entity_fp": ent.get("fp"),
        "entity_fn": ent.get("fn"),
        "entity_precision": ent.get("precision"),
        "entity_recall": ent.get("recall"),
        "entity_f1": ent.get("f1"),
        "relation_tp": rel.get("tp"),
        "relation_fp": rel.get("fp"),
        "relation_fn": rel.get("fn"),
        "relation_precision": rel.get("precision"),
        "relation_recall": rel.get("recall"),
        "relation_f1": rel.get("f1"),
        "pred_entities": counts.get("pred_entities"),
        "gold_entities": counts.get("gold_entities"),
        "pred_relations": counts.get("pred_relations"),
        "gold_relations": counts.get("gold_relations"),
    }

    # Per-relation metrics
    per_relation_rows = []
    for predicate, metrics in extraction.get("per_relation", {}).items():
        per_relation_rows.append({
            "series": series,
            "layer_name": layer_name,
            "stop_index": stop_index,
            "profile": row["profile"],
            "predicate": predicate,
            **metrics,
        })

    return {"summary": summary, "row": row, "per_relation_rows": per_relation_rows}


## 6. Run exact evaluations

In [ ]:
rows = []
per_relation_rows = []
raw_summaries = {}

# Evaluate prefix generated outputs.
for rec in tqdm(prefix_folders, desc="Evaluating prefix folders"):
    result = run_exact_eval_for_folder(
        input_folder=rec["input_folder"],
        out_name=rec["layer_name"],
        run_id=rec["layer_name"],
        series=rec["series"],
        layer_name=rec["layer_name"],
        stop_index=rec["stop_index"],
        triple_count=rec["triple_count"],
    )
    rows.append(result["row"])
    per_relation_rows.extend(result["per_relation_rows"])
    raw_summaries[rec["layer_name"]] = result["summary"]

# Evaluate native final NeoOLAF export with the exact same runner.
native_triples_count = None
native_kg = find_kg_file(NATIVE_EXPORT_DIR)
if native_kg is not None:
    try:
        native_data = json.loads(native_kg.read_text(encoding="utf-8"))
        native_triples_count = len(native_data.get("triples", [])) if isinstance(native_data, dict) else None
    except Exception:
        native_triples_count = None

native_result = run_exact_eval_for_folder(
    input_folder=NATIVE_EXPORT_DIR,
    out_name="native_neoolaf_final_export",
    run_id="native_neoolaf_final_export",
    series="native_neoolaf_final_export",
    layer_name="native_neoolaf_final_export",
    stop_index=12,
    triple_count=native_triples_count,
)
rows.append(native_result["row"])
per_relation_rows.extend(native_result["per_relation_rows"])
raw_summaries["native_neoolaf_final_export"] = native_result["summary"]

summary_df = pd.DataFrame(rows)
per_relation_df = pd.DataFrame(per_relation_rows)

summary_csv = EVAL_OUTPUT_ROOT / "summary_exact_cli_profile_eval.csv"
per_relation_csv = EVAL_OUTPUT_ROOT / "per_relation_exact_cli_profile_eval.csv"
raw_json = EVAL_OUTPUT_ROOT / "raw_summaries_exact_cli_profile_eval.json"

summary_df.to_csv(summary_csv, index=False)
per_relation_df.to_csv(per_relation_csv, index=False)
raw_json.write_text(json.dumps(raw_summaries, indent=2, ensure_ascii=False), encoding="utf-8")

print("Saved:", summary_csv)
print("Saved:", per_relation_csv)
print("Saved:", raw_json)

display(summary_df.sort_values(["series", "stop_index"]))


## 7. Sanity check native final export

This must match the original gold-evaluation notebook. If it does not, the run paths/profile are not the same.


In [ ]:
native_row = summary_df[summary_df["series"] == "native_neoolaf_final_export"].iloc[0]
print(native_row[[
    "profile",
    "triple_count",
    "entity_f1",
    "relation_precision",
    "relation_recall",
    "relation_f1",
    "relation_tp",
    "relation_fp",
    "relation_fn",
    "gold_relations",
]].to_string())

expected_min_f1 = 0.55
if native_row["relation_f1"] < expected_min_f1:
    print("\nWARNING: Native final export relation_f1 is lower than expected.")
    print("Check that DOCUMENT_PROFILE is xquality_relaxed_recall and NATIVE_EXPORT_DIR points to the same exports folder used in the original gold notebook.")
else:
    print("\nOK: Native final export relation F1 is in the expected range.")


## 8. Final comparison table

In [ ]:
cols = [
    "series", "layer_name", "stop_index", "profile", "triple_count",
    "relation_precision", "relation_recall", "relation_f1", "entity_f1",
    "relation_tp", "relation_fp", "relation_fn",
]
final_table = summary_df[cols].sort_values(["series", "stop_index"]).copy()
display(final_table)

print("Best generated prefix by relation_f1:")
display(
    final_table[final_table["series"] == "prefix_stop_after_layer_generated_output"]
    .sort_values("relation_f1", ascending=False)
    .head(10)
)


## 9. Plots

In [ ]:
plot_df = summary_df.copy()
prefix_plot = plot_df[plot_df["series"] == "prefix_stop_after_layer_generated_output"].sort_values("stop_index")
native_plot = plot_df[plot_df["series"] == "native_neoolaf_final_export"]

plt.figure(figsize=(12, 5))
plt.plot(prefix_plot["stop_index"], prefix_plot["relation_f1"], marker="o", label="Prefix stop-after-layer generated output")
if len(native_plot):
    y = float(native_plot.iloc[0]["relation_f1"])
    plt.axhline(y, linestyle="--", label=f"Native NeoOLAF final export ({y:.3f})")
plt.xlabel("Stop-after layer index")
plt.ylabel("Relation F1")
plt.title(f"Exact CLI-compatible evaluation ({DOCUMENT_PROFILE})")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig_path = EVAL_OUTPUT_ROOT / "relation_f1_prefix_vs_native_exact_cli.png"
plt.savefig(fig_path, dpi=180)
plt.show()
print("Saved:", fig_path)

plt.figure(figsize=(12, 5))
plt.plot(prefix_plot["stop_index"], prefix_plot["relation_precision"], marker="o", label="Precision")
plt.plot(prefix_plot["stop_index"], prefix_plot["relation_recall"], marker="o", label="Recall")
plt.plot(prefix_plot["stop_index"], prefix_plot["relation_f1"], marker="o", label="F1")
plt.xlabel("Stop-after layer index")
plt.ylabel("Score")
plt.title(f"Prefix generated outputs: relation metrics ({DOCUMENT_PROFILE})")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig_path = EVAL_OUTPUT_ROOT / "relation_prf_prefix_exact_cli.png"
plt.savefig(fig_path, dpi=180)
plt.show()
print("Saved:", fig_path)


## 10. Interpretation helper

In [ ]:
best_prefix = (
    summary_df[summary_df["series"] == "prefix_stop_after_layer_generated_output"]
    .sort_values("relation_f1", ascending=False)
    .head(1)
)
native = summary_df[summary_df["series"] == "native_neoolaf_final_export"].head(1)

if len(best_prefix) and len(native):
    bp = best_prefix.iloc[0]
    nr = native.iloc[0]
    print("Best generated prefix:")
    print(f"  {bp['layer_name']} | Relation P/R/F1 = {bp['relation_precision']:.3f}/{bp['relation_recall']:.3f}/{bp['relation_f1']:.3f}")
    print("Native NeoOLAF final export:")
    print(f"  Relation P/R/F1 = {nr['relation_precision']:.3f}/{nr['relation_recall']:.3f}/{nr['relation_f1']:.3f}")
    print()
    if nr["relation_f1"] > bp["relation_f1"]:
        print("Interpretation: the native NeoOLAF finalization outperforms the prefix-generated stop-after-layer outputs.")
    else:
        print("Interpretation: one prefix-generated output outperforms the native final export; inspect over-generation, precision, and fairness.")
